In [1]:
!nvidia-smi

Tue Sep  8 04:19:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
%%writefile matrix_mul.cu

#include <cuda_runtime.h>
#include <chrono>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <vector>
#include <cmath>

#define TILE 16

__global__ void matrixMulKernel(
    const float* A,
    const float* B,
    float* C,
    int N
) {
    __shared__ float tileA[TILE][TILE];
    __shared__ float tileB[TILE][TILE];

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    float sum = 0.0f;

    int numberOfTiles = (N + TILE - 1) / TILE;

    for (int t = 0; t < numberOfTiles; ++t) {

        int A_col = t * TILE + threadIdx.x;
        int B_row = t * TILE + threadIdx.y;

        if (row < N && A_col < N)
            tileA[threadIdx.y][threadIdx.x] =
                A[row * N + A_col];
        else
            tileA[threadIdx.y][threadIdx.x] = 0.0f;

        if (B_row < N && col < N)
            tileB[threadIdx.y][threadIdx.x] =
                B[B_row * N + col];
        else
            tileB[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();

        for (int k = 0; k < TILE; ++k) {
            sum +=
                tileA[threadIdx.y][k] *
                tileB[k][threadIdx.x];
        }

        __syncthreads();
    }

    if (row < N && col < N)
        C[row * N + col] = sum;
}


void cpuMatrixMultiply(
    const float* A,
    const float* B,
    float* C,
    int N
) {
    for (int i = 0; i < N; ++i) {
        for (int k = 0; k < N; ++k) {

            float value = A[i * N + k];

            for (int j = 0; j < N; ++j) {
                C[i * N + j] +=
                    value * B[k * N + j];
            }
        }
    }
}


void runBenchmark(int N) {

    size_t elements =
        static_cast<size_t>(N) * N;

    size_t bytes =
        elements * sizeof(float);

    std::vector<float> A(elements);
    std::vector<float> B(elements);

    std::vector<float> C_cpu(
        elements,
        0.0f
    );

    std::vector<float> C_gpu(
        elements,
        0.0f
    );

    srand(5996);

    for (size_t i = 0; i < elements; ++i) {

        A[i] =
            static_cast<float>(rand()) /
            RAND_MAX;

        B[i] =
            static_cast<float>(rand()) /
            RAND_MAX;
    }

    // CPU timing
    auto cpuStart =
        std::chrono::high_resolution_clock::now();

    cpuMatrixMultiply(
        A.data(),
        B.data(),
        C_cpu.data(),
        N
    );

    auto cpuEnd =
        std::chrono::high_resolution_clock::now();

    double cpuMs =
        std::chrono::duration<double, std::milli>(
            cpuEnd - cpuStart
        ).count();


    // GPU memory
    float* d_A;
    float* d_B;
    float* d_C;

    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);


    cudaEvent_t start;
    cudaEvent_t stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);


    // Host to Device
    cudaEventRecord(start);

    cudaMemcpy(
        d_A,
        A.data(),
        bytes,
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_B,
        B.data(),
        bytes,
        cudaMemcpyHostToDevice
    );

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float h2dMs;

    cudaEventElapsedTime(
        &h2dMs,
        start,
        stop
    );


    dim3 threadsPerBlock(
        TILE,
        TILE
    );

    dim3 blocksPerGrid(
        (N + TILE - 1) / TILE,
        (N + TILE - 1) / TILE
    );


    // Warm up
    matrixMulKernel<<<
        blocksPerGrid,
        threadsPerBlock
    >>>(
        d_A,
        d_B,
        d_C,
        N
    );

    cudaDeviceSynchronize();


    // Kernel timing
    cudaEventRecord(start);

    matrixMulKernel<<<
        blocksPerGrid,
        threadsPerBlock
    >>>(
        d_A,
        d_B,
        d_C,
        N
    );

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float kernelMs;

    cudaEventElapsedTime(
        &kernelMs,
        start,
        stop
    );


    // Device to Host
    cudaEventRecord(start);

    cudaMemcpy(
        C_gpu.data(),
        d_C,
        bytes,
        cudaMemcpyDeviceToHost
    );

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float d2hMs;

    cudaEventElapsedTime(
        &d2hMs,
        start,
        stop
    );


    float transferMs =
        h2dMs + d2hMs;

    float gpuEndToEndMs =
        kernelMs + transferMs;

    double speedup =
        cpuMs / gpuEndToEndMs;


    // Correctness check
    float maxError = 0.0f;

    for (size_t i = 0; i < elements; ++i) {

        float error =
            std::fabs(
                C_cpu[i] - C_gpu[i]
            );

        if (error > maxError)
            maxError = error;
    }


    std::cout
        << std::fixed
        << std::setprecision(4);

    std::cout << "\nMatrix size: "
              << N << " x " << N
              << std::endl;

    std::cout << "CPU time (ms): "
              << cpuMs
              << std::endl;

    std::cout << "GPU kernel time (ms): "
              << kernelMs
              << std::endl;

    std::cout << "H2D time (ms): "
              << h2dMs
              << std::endl;

    std::cout << "D2H time (ms): "
              << d2hMs
              << std::endl;

    std::cout << "H2D+D2H (ms): "
              << transferMs
              << std::endl;

    std::cout << "GPU end-to-end (ms): "
              << gpuEndToEndMs
              << std::endl;

    std::cout << "End-to-end speedup: "
              << speedup
              << "x"
              << std::endl;

    std::cout << "Maximum error: "
              << maxError
              << std::endl;


    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
}


int main() {

    runBenchmark(256);
    runBenchmark(1024);
    runBenchmark(4096);

    return 0;
}

Writing matrix_mul.cu


In [5]:
!nvcc -O3 matrix_mul.cu -o matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
!./matrix_mul


Matrix size: 256 x 256
CPU time (ms): 2.4566
GPU kernel time (ms): 0.0735
H2D time (ms): 1.0151
D2H time (ms): 0.1259
H2D+D2H (ms): 1.1410
GPU end-to-end (ms): 1.2144
End-to-end speedup: 2.0229x
Maximum error: 0.0000

Matrix size: 1024 x 1024
CPU time (ms): 202.2910
GPU kernel time (ms): 3.6663
H2D time (ms): 2.0033
D2H time (ms): 1.0431
H2D+D2H (ms): 3.0464
GPU end-to-end (ms): 6.7127
End-to-end speedup: 30.1355x
Maximum error: 0.0001

Matrix size: 4096 x 4096
CPU time (ms): 20196.8277
GPU kernel time (ms): 193.8336
H2D time (ms): 29.6901
D2H time (ms): 16.0351
H2D+D2H (ms): 45.7253
GPU end-to-end (ms): 239.5589
End-to-end speedup: 84.3084x
Maximum error: 0.0004


## CUDA Blocks and Threads

The CUDA kernel uses a two-dimensional block containing 16 × 16
threads, for a total of 256 threads per block. Each thread computes
one element of the output matrix.

The output row and column assigned to each thread are calculated using
the block index and thread index:

`row = blockIdx.y * blockDim.y + threadIdx.y`

`col = blockIdx.x * blockDim.x + threadIdx.x`

The grid dimensions are calculated so that enough blocks are launched
to cover the complete N × N matrix.

The kernel also uses 16 × 16 shared-memory tiles for matrices A and B.
Threads within a block cooperatively load matrix values into shared
memory and reuse those values during the dot-product calculation.
This reduces repeated global-memory accesses. The `__syncthreads()`
calls ensure the shared tiles are fully loaded before computation
continues.

In [7]:
!which nsys
!which ncu
!which nvprof

/usr/local/cuda/bin/ncu
/usr/local/cuda/bin/nvprof


In [8]:
!ncu --target-processes all ./matrix_mul

==PROF== Connected to process 2020 (/content/matrix_mul)
==PROF== Profiling "matrixMulKernel" - 0: 0%....50%....100% - 9 passes
==PROF== Profiling "matrixMulKernel" - 1: 0%....50%....100% - 9 passes

Matrix size: 256 x 256
CPU time (ms): 2.0500
GPU kernel time (ms): 1606.7201
H2D time (ms): 0.2163
D2H time (ms): 0.1531
H2D+D2H (ms): 0.3693
GPU end-to-end (ms): 1607.0894
End-to-end speedup: 0.0013x
Maximum error: 0.0000
==PROF== Profiling "matrixMulKernel" - 2: 0%....50%....100% - 9 passes
==PROF== Profiling "matrixMulKernel" - 3: 0%....50%....100% - 9 passes

Matrix size: 1024 x 1024
CPU time (ms): 274.8628
GPU kernel time (ms): 1724.5001
H2D time (ms): 2.0301
D2H time (ms): 1.1253
H2D+D2H (ms): 3.1555
GPU end-to-end (ms): 1727.6555
End-to-end speedup: 0.1591x
Maximum error: 0.0001
==PROF== Profiling "matrixMulKernel" - 4: 0%....50%....100% - 9 passes
==PROF== Profiling "matrixMulKernel" - 5: 0%....50%....100% - 9 passes

Matrix size: 4096 x 4096
CPU time (ms): 24020.4603
GPU kernel ti

## CUDA Timing Results

| Matrix Size (N) | CPU Time (ms) | GPU Kernel Time (ms) | H2D + D2H (ms) | GPU End-to-End Time (ms) | End-to-End Speedup |
|---:|---:|---:|---:|---:|---:|
| 256  | 2.0500 | 1606.7201 | 0.3693 | 1607.0894 | 0.0013x |
| 1024 | 274.8628 | 1724.5001 | 3.1555 | 1727.6555 | 0.1591x |
| 4096 | 24020.4603 | 4960.0088 | 52.6253 | 5012.6343 | 4.7920x |

**Profiler used:** NVIDIA Nsight Compute (`ncu`)

**Speedup calculation:**  
`CPU Time / GPU End-to-End Time`

where:

`GPU End-to-End Time = GPU Kernel Time + H2D Time + D2H Time`

### CUDA Profiling

The CUDA program was profiled using **NVIDIA Nsight Compute (`ncu`)**
on an NVIDIA Tesla T4 GPU. The profiler confirmed a block size of
256 threads, corresponding to the 16 × 16 thread-block configuration
used by the kernel.

For the largest matrix size, the profiler reported high achieved
occupancy and showed substantial GPU utilization during matrix
multiplication. The profiler also indicated that computation and memory
traffic were relatively well balanced. Host-to-device and
device-to-host transfer times were measured separately using CUDA
events in the program.

### Correctness Verification

The CUDA results were compared element-by-element against the CPU
matrix multiplication results. The maximum absolute errors were
0.0000 for N=256, 0.0001 for N=1024, and 0.0004 for N=4096.
These small differences are consistent with floating-point rounding
effects caused by different orders of arithmetic operations on the
CPU and GPU. Therefore, the CUDA implementation produced numerically
consistent results with the CPU implementation.


### CPU-GPU Crossover Analysis

For the measured matrix sizes, the GPU became faster than the CPU at
N=4096, where the end-to-end speedup was approximately 4.79×. At
N=256 and N=1024, the GPU overhead was too large relative to the amount
of computation, resulting in speedups below 1. As matrix size increases,
the much larger amount of parallel computation allows the GPU to
amortize kernel and data-transfer overhead, making GPU execution more
beneficial.


In [9]:
!./matrix_mul


Matrix size: 256 x 256
CPU time (ms): 2.1230
GPU kernel time (ms): 0.0620
H2D time (ms): 0.2558
D2H time (ms): 0.0936
H2D+D2H (ms): 0.3494
GPU end-to-end (ms): 0.4115
End-to-end speedup: 5.1594x
Maximum error: 0.0000

Matrix size: 1024 x 1024
CPU time (ms): 193.0717
GPU kernel time (ms): 3.1045
H2D time (ms): 2.0077
D2H time (ms): 1.0578
H2D+D2H (ms): 3.0655
GPU end-to-end (ms): 6.1700
End-to-end speedup: 31.2918x
Maximum error: 0.0001

Matrix size: 4096 x 4096
CPU time (ms): 19693.2230
GPU kernel time (ms): 200.9600
H2D time (ms): 28.8108
D2H time (ms): 15.1544
H2D+D2H (ms): 43.9652
GPU end-to-end (ms): 244.9252
End-to-end speedup: 80.4050x
Maximum error: 0.0004


## CUDA Timing Results - Normal Execution

| Matrix Size (N) | CPU Time (ms) | GPU Kernel Time (ms) | H2D + D2H Time (ms) | GPU End-to-End Time (ms) | End-to-End Speedup |
|---:|---:|---:|---:|---:|---:|
| 256  | 2.1230 | 0.0620 | 0.3494 | 0.4115 | 5.1594× |
| 1024 | 193.0717 | 3.1045 | 3.0655 | 6.1700 | 31.2918× |
| 4096 | 19693.2230 | 200.9600 | 43.9652 | 244.9252 | 80.4050× |

GPU end-to-end time includes the CUDA kernel execution time and
host-to-device/device-to-host transfer time.

End-to-end speedup was calculated as:

`CPU Time / GPU End-to-End Time`

### CUDA Profiling

The CUDA kernel was profiled using **NVIDIA Nsight Compute (`ncu`)**
on an NVIDIA Tesla T4 GPU. Nsight Compute confirmed that the kernel
used a block size of 256 threads, corresponding to the 16 × 16
thread-block configuration.

The profiler also provided information about occupancy, compute
throughput, memory behavior, and kernel launch characteristics.
The execution times observed while profiling were substantially higher
than the normal execution times because Nsight Compute instruments and
replays the kernel across multiple profiling passes. Therefore, the
normal `./matrix_mul` execution results are used for the primary timing
and speedup comparison.

### Correctness Verification

| Matrix Size (N) | Maximum Absolute Error |
|---:|---:|
| 256 | 0.0000 |
| 1024 | 0.0001 |
| 4096 | 0.0004 |

The GPU output was compared element-by-element with the CPU result.
The maximum absolute errors remained very small for all three matrix
sizes, indicating that the CUDA implementation produced numerically
consistent results. The small differences are expected from
floating-point rounding and differences in the order of arithmetic
operations between CPU and GPU execution.


### CPU-GPU Crossover Discussion

The GPU was already faster than the CPU at the smallest tested matrix
size, N=256, with an end-to-end speedup of approximately 5.16×.
The speedup increased to approximately 31.29× for N=1024 and 80.41×
for N=4096. As matrix size increases, more matrix operations can be
performed concurrently on the GPU, allowing its parallel architecture
to be utilized more effectively. The increasing computation also
amortizes the relative cost of host-device memory transfers, producing
larger speedups for the larger matrices.


### CUDA Blocks and Threads

The CUDA kernel uses two-dimensional thread blocks of 16 × 16 threads,
giving 256 threads per block. Each thread is responsible for computing
one element of the output matrix C.

The grid dimensions are calculated as `ceil(N/16) × ceil(N/16)` so
that all output elements are covered. For example, N=4096 uses a
256 × 256 grid containing 65,536 blocks.

The kernel uses 16 × 16 shared-memory tiles for portions of matrices A
and B. Threads within each block cooperatively load these tiles into
shared memory and reuse the data while computing the output values,
reducing repeated accesses to global GPU memory.

In [10]:
for run in range(3):
    print(f"\n========== RUN {run + 1} ==========")
    !./matrix_mul


========== RUN 1 ==========

Matrix size: 256 x 256
CPU time (ms): 2.0488
GPU kernel time (ms): 0.0691
H2D time (ms): 0.1878
D2H time (ms): 0.0994
H2D+D2H (ms): 0.2872
GPU end-to-end (ms): 0.3563
End-to-end speedup: 5.7499x
Maximum error: 0.0000

Matrix size: 1024 x 1024
CPU time (ms): 202.7078
GPU kernel time (ms): 3.4471
H2D time (ms): 1.9824
D2H time (ms): 1.0238
H2D+D2H (ms): 3.0062
GPU end-to-end (ms): 6.4533
End-to-end speedup: 31.4114x
Maximum error: 0.0001

Matrix size: 4096 x 4096
CPU time (ms): 20285.3755
GPU kernel time (ms): 198.2307
H2D time (ms): 32.2907
D2H time (ms): 17.2745
H2D+D2H (ms): 49.5652
GPU end-to-end (ms): 247.7958
End-to-end speedup: 81.8633x
Maximum error: 0.0004

========== RUN 2 ==========

Matrix size: 256 x 256
CPU time (ms): 5.2090
GPU kernel time (ms): 0.0446
H2D time (ms): 0.1830
D2H time (ms): 0.0925
H2D+D2H (ms): 0.2755
GPU end-to-end (ms): 0.3201
End-to-end speedup: 16.2731x
Maximum error: 0.0000

Matrix size: 1024 x 1024
CPU time (ms): 199.1947


## Final CUDA Timing Results

Each matrix size was measured three times after GPU warm-up. The table
reports the mean execution time across the three measured runs. The
end-to-end speedup is calculated using the mean CPU time divided by the
mean GPU end-to-end time.

| Matrix Size (N) | Mean CPU Time (ms) | Mean GPU Kernel Time (ms) | Mean H2D + D2H (ms) | Mean GPU End-to-End (ms) | End-to-End Speedup |
|---:|---:|---:|---:|---:|---:|
| 256  | 3.0701 | 0.0541 | 0.2788 | 0.3330 | 9.22× |
| 1024 | 199.7894 | 2.6487 | 2.9977 | 5.6464 | 35.39× |
| 4096 | 20256.7097 | 199.5739 | 47.0412 | 246.6151 | 82.14× |

**GPU:** NVIDIA Tesla T4  
**Number of measured runs:** 3  
**Aggregation:** Arithmetic mean  
**Profiler:** NVIDIA Nsight Compute (`ncu`)

`GPU End-to-End Time = GPU Kernel Time + H2D Time + D2H Time`

`End-to-End Speedup = Mean CPU Time / Mean GPU End-to-End Time`

### CPU-GPU Crossover Discussion

The GPU was already faster than the CPU at the smallest tested matrix
size, N=256, with an average end-to-end speedup of approximately 9.22×.
The speedup increased to approximately 35.39× for N=1024 and 82.14×
for N=4096. As the matrix size increased, the larger amount of
parallel computation allowed the GPU to utilize its processing
resources more effectively. The cost of host-device memory transfers
also became smaller relative to the amount of computation, resulting
in greater GPU benefits for larger matrices.

### Correctness Verification

| Matrix Size (N) | Maximum Absolute Error |
|---:|---:|
| 256 | 0.0000 |
| 1024 | 0.0001 |
| 4096 | 0.0004 |

The CUDA output was compared element-by-element against the CPU matrix
multiplication result. The maximum absolute error remained very small
for all tested matrix sizes, confirming that the CUDA implementation
produced numerically consistent results. The small differences are
expected because floating-point operations may be evaluated in a
different order on the CPU and GPU.

### CUDA Profiling

NVIDIA Nsight Compute (`ncu`) was used to profile the CUDA matrix
multiplication kernel on the Tesla T4 GPU. The profiler confirmed a
block size of 256 threads, corresponding to the 16 × 16 thread-block
configuration used by the kernel. For the largest workload, the
profiler reported approximately 99.9% achieved occupancy and indicated
that compute and memory activity were relatively well balanced.

The execution times produced while running under Nsight Compute were
not used as the primary benchmark measurements because the profiler
instruments and replays the kernel across multiple profiling passes.
The normal repeated executions of `./matrix_mul` were therefore used
for the final timing and speedup measurements.